In [1]:
# ============================================================
# CÉLULA 1 — Imports e carregamento
# ============================================================
from pathlib import Path

import numpy as np
import pandas as pd
from sklearn.ensemble import RandomForestRegressor
from sklearn.linear_model import Ridge
from sklearn.metrics import mean_absolute_error, mean_squared_error
from sklearn.preprocessing import StandardScaler

RAIZ_PROJETO = Path.cwd().parent
base_final = pd.read_parquet(RAIZ_PROJETO / "data" / "processed" / "base_inmet_sidra.parquet")
base_final = base_final.sort_values(['code_micro', 'ano']).reset_index(drop=True)

# Ajuste esta lista para os nomes reais das suas colunas climáticas
COLUNAS_CLIMA = [
    'prec_total_safra_completa', 'temp_media_safra_completa',
    'temp_max_abs_safra_completa', 'temp_min_abs_safra_completa',
    'gdd_acumulado_safra_completa', 'dias_secos_safra_completa',
]


# ============================================================
# CÉLULA 2 — Feature engineering: lag de produtividade
# ============================================================
# Captura tendência tecnológica (produtividade sobe ano a ano por
# melhoria genética/manejo, independente do clima) e o nível de base de
# cada microrregião -- sem isso, o modelo pode confundir "ano recente"
# ou "microrregião historicamente produtiva" com "clima favorável".
base_final['rendimento_lag1'] = base_final.groupby('code_micro')['rendimento_kg_ha'].shift(1)

base_modelo = base_final.dropna(subset=['rendimento_lag1'] + COLUNAS_CLIMA).copy()
print(f"Shape após remover NaN (lag + clima): {base_modelo.shape} "
      f"(de {base_final.shape[0]} linhas originais)")

# Checagem de multicolinearidade -- comum entre temp_media e
# gdd_acumulado, que carregam informação parecida
print("\nCorrelação entre variáveis climáticas:")
print(base_modelo[COLUNAS_CLIMA].corr().round(2))


# ============================================================
# CÉLULA 3 — Separação temporal (NUNCA aleatória em dado em painel)
# ============================================================
# Um split aleatório vazaria informação do futuro para o passado dentro
# da mesma microrregião. O corte é por ano: treina no passado, testa no
# que viria "depois" -- simula a situação real de previsão.
ANO_CORTE = base_modelo['ano'].quantile(0.8, interpolation='lower')  # ~últimos 20% dos anos como teste
print(f"Ano de corte escolhido: {ANO_CORTE} (ajuste manualmente se preferir um valor fixo)")

treino = base_modelo[base_modelo['ano'] < ANO_CORTE]
teste = base_modelo[base_modelo['ano'] >= ANO_CORTE]
print(f"Treino: {treino.shape[0]} linhas | Teste: {teste.shape[0]} linhas")


def avaliar(y_true, y_pred, nome):
    mae = mean_absolute_error(y_true, y_pred)
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    print(f"{nome:40s} MAE={mae:8.1f} kg/ha | RMSE={rmse:8.1f} kg/ha")
    return {'modelo': nome, 'mae': mae, 'rmse': rmse}


resultados = []


# ============================================================
# CÉLULA 4 — Baseline: persistência
# ============================================================
# A régua mínima: prever "igual ao ano anterior". Se nenhum modelo
# superar isso, o clima não está agregando valor real à previsão.
resultados.append(avaliar(teste['rendimento_kg_ha'], teste['rendimento_lag1'], "Baseline (persistência)"))


# ============================================================
# CÉLULA 5 — Regressão linear regularizada com efeito fixo de microrregião
# ============================================================
# code_micro como dummy: isola o efeito do clima DENTRO de cada região
# ao longo do tempo, em vez de confundir com diferenças de nível entre
# regiões (solo, tecnologia local, etc).
features_numericas = ['rendimento_lag1'] + COLUNAS_CLIMA
dummies_micro = pd.get_dummies(base_modelo['code_micro'], prefix='micro', drop_first=True)

X = pd.concat([base_modelo[features_numericas], dummies_micro], axis=1)
y = base_modelo['rendimento_kg_ha']

X_treino, X_teste = X.loc[treino.index], X.loc[teste.index]
y_treino, y_teste = y.loc[treino.index], y.loc[teste.index]

scaler = StandardScaler()
X_treino_esc = X_treino.copy()
X_teste_esc = X_teste.copy()
X_treino_esc[features_numericas] = scaler.fit_transform(X_treino[features_numericas])
X_teste_esc[features_numericas] = scaler.transform(X_teste[features_numericas])

modelo_linear = Ridge(alpha=1.0)
modelo_linear.fit(X_treino_esc, y_treino)
pred_linear = modelo_linear.predict(X_teste_esc)
resultados.append(avaliar(y_teste, pred_linear, "Ridge (efeito fixo + clima + lag)"))

coefs = pd.Series(modelo_linear.coef_, index=X.columns)
print("\nCoeficientes das variáveis numéricas (já padronizadas) -- confira")
print("o SINAL contra o esperado agronomicamente antes de confiar no modelo:")
print(coefs[features_numericas].sort_values(ascending=False))


# ============================================================
# CÉLULA 6 — Random Forest
# ============================================================
# Alternativa a XGBoost/LightGBM sem precisar instalar mais nada; troque
# por xgboost.XGBRegressor ou lightgbm.LGBMRegressor se preferir e tiver
# instalado. Árvores não precisam de padronização de escala.
modelo_rf = RandomForestRegressor(n_estimators=300, max_depth=6, random_state=42)
modelo_rf.fit(X_treino, y_treino)
pred_rf = modelo_rf.predict(X_teste)
resultados.append(avaliar(y_teste, pred_rf, "Random Forest (efeito fixo + clima + lag)"))

print("\nImportância das features (Random Forest):")
print(pd.Series(modelo_rf.feature_importances_, index=X.columns)[features_numericas]
      .sort_values(ascending=False))


# ============================================================
# CÉLULA 7 — Comparação final
# ============================================================
comparacao = pd.DataFrame(resultados).sort_values('mae')
print("\n", comparacao.to_string(index=False))


# ============================================================
# CÉLULA 8 (opcional) — Modelo misto: code_micro como efeito ALEATÓRIO
# ============================================================
# Alternativa mais apropriada que dummies fixas se muitas microrregiões
# tiverem poucos anos de histórico -- o efeito aleatório "encolhe" a
# estimativa de regiões com pouco dado em direção à média geral, em vez
# de dar a elas um parâmetro totalmente livre como a dummy faria.
# Requer `pip install statsmodels` -- não testado neste ambiente.
#
# import statsmodels.formula.api as smf
#
# formula = "rendimento_kg_ha ~ rendimento_lag1 + " + " + ".join(COLUNAS_CLIMA)
# modelo_misto = smf.mixedlm(formula, data=treino, groups=treino['code_micro'])
# resultado_misto = modelo_misto.fit()
# print(resultado_misto.summary())

Shape após remover NaN (lag + clima): (3488, 20) (de 4413 linhas originais)

Correlação entre variáveis climáticas:
                              prec_total_safra_completa  \
prec_total_safra_completa                          1.00   
temp_media_safra_completa                         -0.10   
temp_max_abs_safra_completa                       -0.19   
temp_min_abs_safra_completa                       -0.02   
gdd_acumulado_safra_completa                      -0.07   
dias_secos_safra_completa                         -0.78   

                              temp_media_safra_completa  \
prec_total_safra_completa                         -0.10   
temp_media_safra_completa                          1.00   
temp_max_abs_safra_completa                        0.77   
temp_min_abs_safra_completa                        0.83   
gdd_acumulado_safra_completa                       0.96   
dias_secos_safra_completa                          0.08   

                              temp_max_abs_safra_complet